In [3]:
import pandas as pd
df=pd.read_csv("adaptive_run.csv")
df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 999 entries, 0 to 998
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   timestamp        999 non-null    float64
 1   input_id         999 non-null    int64  
 2   cpu_raw_before   999 non-null    float64
 3   cpu_ema_before   999 non-null    float64
 4   mem_raw_before   999 non-null    float64
 5   mem_ema_before   999 non-null    float64
 6   temp_before      999 non-null    float64
 7   cpu_raw_after    999 non-null    float64
 8   cpu_ema_after    999 non-null    float64
 9   mem_raw_after    999 non-null    float64
 10  mem_ema_after    999 non-null    float64
 11  temp_after       999 non-null    float64
 12  chosen_action    999 non-null    int64  
 13  model_used       999 non-null    object 
 14  predicted_class  999 non-null    int64  
 15  label            999 non-null    object 
 16  confidence       999 non-null    float64
 17  latency_ms      

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 999 entries, 0 to 998
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   timestamp        999 non-null    float64
 1   input_id         999 non-null    int64  
 2   cpu_raw_before   999 non-null    float64
 3   cpu_ema_before   999 non-null    float64
 4   mem_raw_before   999 non-null    float64
 5   mem_ema_before   999 non-null    float64
 6   temp_before      999 non-null    float64
 7   cpu_raw_after    999 non-null    float64
 8   cpu_ema_after    999 non-null    float64
 9   mem_raw_after    999 non-null    float64
 10  mem_ema_after    999 non-null    float64
 11  temp_after       999 non-null    float64
 12  chosen_action    999 non-null    int64  
 13  model_used       999 non-null    object 
 14  predicted_class  999 non-null    int64  
 15  label            999 non-null    object 
 16  confidence       999 non-null    float64
 17  latency_ms      

In [5]:
import os
import pandas as pd
import matplotlib.pyplot as plt

CSV_FILE = "adaptive_run.csv"

os.makedirs("figures", exist_ok=True)

df = pd.read_csv(CSV_FILE)

print("\n========== DATASET SUMMARY ==========\n")

print(f"Rows: {len(df)}")
print()

print(df.describe())

print("\nAction Counts:")
print(df["model_used"].value_counts())

print("\nAverage Metrics by Model:")
print(
    df.groupby("model_used")[
        ["latency_ms", "reward"]
    ].mean()
)

# -------------------------
# Action Distribution
# -------------------------

plt.figure(figsize=(6,4))
df["model_used"].value_counts().plot(kind="bar")
plt.title("Action Distribution")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("figures/action_distribution.png")
plt.close()

# -------------------------
# CPU Distribution
# -------------------------

plt.figure(figsize=(8,4))
plt.hist(df["cpu_ema_before"], bins=30)
plt.title("CPU EMA Distribution")
plt.xlabel("CPU EMA")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig("figures/cpu_distribution.png")
plt.close()

# -------------------------
# Latency by Model
# -------------------------

plt.figure(figsize=(6,4))

df.boxplot(
    column="latency_ms",
    by="model_used"
)

plt.title("Latency by Model")
plt.suptitle("")
plt.ylabel("Latency (ms)")
plt.tight_layout()
plt.savefig("figures/latency_by_model.png")
plt.close()

# -------------------------
# Reward by Model
# -------------------------

plt.figure(figsize=(6,4))

df.boxplot(
    column="reward",
    by="model_used"
)

plt.title("Reward by Model")
plt.suptitle("")
plt.ylabel("Reward")
plt.tight_layout()
plt.savefig("figures/reward_by_model.png")
plt.close()

# -------------------------
# CPU vs Latency
# -------------------------

plt.figure(figsize=(8,5))

for model in df["model_used"].unique():

    subset = df[
        df["model_used"] == model
    ]

    plt.scatter(
        subset["cpu_ema_before"],
        subset["latency_ms"],
        alpha=0.5,
        label=model
    )

plt.xlabel("CPU EMA")
plt.ylabel("Latency (ms)")
plt.title("CPU Load vs Latency")
plt.legend()
plt.tight_layout()
plt.savefig("figures/cpu_vs_latency.png")
plt.close()

# -------------------------
# Summary File
# -------------------------

with open("figures/summary.txt", "w") as f:

    f.write(
        f"Rows: {len(df)}\n\n"
    )

    f.write("Action Counts\n")
    f.write(
        str(
            df["model_used"]
            .value_counts()
        )
    )

    f.write("\n\n")

    f.write("Average Metrics\n")
    f.write(
        str(
            df.groupby("model_used")[
                ["latency_ms", "reward"]
            ].mean()
        )
    )

print("\nDone.")
print("Charts saved in figures/")


========== DATASET SUMMARY ==========

Rows: 999

          timestamp    input_id  cpu_raw_before  cpu_ema_before  \
count  9.990000e+02  999.000000      999.000000      999.000000   
mean   1.783701e+09  499.000000       26.625626       26.516229   
std    9.443730e+02  288.530761       20.100632       19.664005   
min    1.783699e+09    0.000000        0.000000        0.051306   
25%    1.783700e+09  249.500000        3.000000        3.204059   
50%    1.783701e+09  499.000000       40.200000       40.393767   
75%    1.783701e+09  748.500000       42.400000       42.731252   
max    1.783702e+09  998.000000       73.600000       58.249431   

       mem_raw_before  mem_ema_before  temp_before  cpu_raw_after  \
count      999.000000      999.000000   999.000000     999.000000   
mean        39.144244       39.160619    57.008158      26.482583   
std          5.887613        5.861809     2.683172      20.160242   
min         25.400000       25.408071    52.850000       0.000000   


<Figure size 600x400 with 0 Axes>

<Figure size 600x400 with 0 Axes>